# Locating gauge stations on MSC model grids



This tutorial demonstrates how to find gauge stations on the NSRPS model grid using the Web Coverage Service (WCS). If this is your first time accessing GeoMet's Web Coverage Service, we recommend first reading the geomet-auth-wcs-quick-demo tutorial. For more detailed information about accessing secured layers with the Web Map Service and Web Coverage Service we recommend reading our more comprehensive tutorials, geomet-auth-wms-examples and geomet-auth-wms-examples.

## General requirements

In [1]:
# data
import re
import configparser
from datetime import datetime, timedelta, timezone
import xarray as xr 
import pandas as pd
import numpy as np
import geopandas as gpd

# web map services 
from owslib.wms import WebMapService
from owslib.wcs import WebCoverageService
from owslib.wcs import Authentication
from owslib.ogcapi.features import Features

# plotting
import holoviews
import hvplot.xarray
import hvplot.pandas

In [2]:
# loading login information
config = configparser.ConfigParser()
config.read_file(open('config.cfg')) 

login = config['Login']

## Gathering information on the latest DHPS forecast

In [3]:
# Deterministic Hydrologic Prediction System (DHPS) is a sub-system of the NSRPS
layer_name_fcast = 'DHPS_1km_RiverDischarge'

In [ ]:
# first querying the WMS for time metadata
wms = WebMapService(
    f'https://geo.weather.gc.ca/geomet?&SERVICE=WMS&LAYERS={layer_name_fcast}',
    version='1.3.0',
    auth=Authentication(username=login['Username'], password=login['Password']),
    timeout=300
)

In [5]:
# Grab forecast time metadata from the WMS
oldest_fcast, newest_fcast, issue_interval = wms[layer_name_fcast].dimensions['reference_time']['values'][0].split('/')
# by default, the first and last times from the wms 'time' attribute belong to the latest_fcast issue
first_datetime, last_datetime, datetime_interval = wms[layer_name_fcast].dimensions['time']['values'][0].split('/')

print(wms[layer_name_fcast].dimensions['time']['values'][0].split('/'))

['2025-08-05T01:00:00Z', '2025-08-11T00:00:00Z', 'PT1H']


In [6]:
iso_format = "%Y-%m-%dT%H:%M:%SZ"

# convert dates to datetime objects
first = datetime.strptime(first_datetime, iso_format)
last = datetime.strptime(last_datetime, iso_format)

# remove anything that isn't a number from the datetime interval (time between forecasts)
intvl = int(re.sub(r'\D', '', datetime_interval))

# create a list of forecast datetimes (we will add these to the requested data)
fcasthrs = [first]
while first < last:
    first = first + timedelta(hours=intvl)
    fcasthrs.append(first)

# create a list of iso formatted forecast datetime strings (we will use these in the WCS requests)
fcasthrs_str = [datetime.strftime(hr, iso_format) for hr in fcasthrs]

## Finding stations on the DHPS model grid

In other tutorials, we've looked at the gridded DHPS data, but we may be more interested in the streamflow forecasts at specific stations. As an example, let's say we are interested in the flows at [Sturgeon River at McDougall Mills](https://wateroffice.ec.gc.ca/report/real_time_e.html?stn=05QA004) (station number 05QA004). The following example demonstrates how we can extract the river discharge forecasts at our station of interest.


Note: the station latitutdes and longitudes in the models (DHPS/WCPS) may not be exactly the same as the actual stations' latitudes and longitudes. In some cases, the modellers have had to slightly adjust the stations' latitudes/longitudes to minimize drainage area errors. This is a by-product of creating a gridded stream network. As a result, to find the stations of interest on the model grid we need to know what latitudes/longitudes the modellers have assigned to each station. In addition, the model's station lat/lon may not align perfectly with the lat/lon axes of the model grid. The lat/lon axes of the model grid specify the center point of each grid cell and a station's model location may not be located in the center of its grid cell. Therefore, to find the closest grid cell that to our station of interest we need to identify the closest lat/lon center point to the model's station lat/lon.

In order to see all stations available and their latitude/longitude on the model grid, we can open the file `nsrps_stn_locations.csv`. In this file, stations are labelled according to station number. Regions in file: Bay of Fundy, Churchill River, Columbia River, Great Lakes - St. Lawrence River, Gulf of St. Lawrence, Mackenzie River, Nelson River, Skeena River, and Yukon River.

In [7]:
# define location of csv and open as a dataframe
station_df = pd.read_csv('../nsrps_stn_locations.csv', skiprows=2) #skipping the first two rows are these are user info
station_df.head()

,STATION_NUMBER,PROVIDER,MAJOR_DRAINAGE_BASIN,MODEL_LATITUDE,MODEL_LONGITUDE
0,05AA008,AEP,Nelson River,49.6066,-114.4093
1,05AA022,AEP,Nelson River,49.4897,-114.1508
2,05AA024,AEP,Nelson River,49.5563,-113.8259
3,05AA035,AEP,Nelson River,49.7315,-114.0841
4,05AB021,AEP,Nelson River,50.0231,-113.7095


In [8]:
# define the station number of interest
stn_num = '05QA004'

# select station of interest
station_df[station_df["STATION_NUMBER"] == stn_num]

,STATION_NUMBER,PROVIDER,MAJOR_DRAINAGE_BASIN,MODEL_LATITUDE,MODEL_LONGITUDE
100,05QA004,ECCC,Nelson River,50.1721,-91.5427


In this example, Sturgeon River at McDougall Mills has an actual lat/lon of 50.167222, -91.540556 (in decimal degrees). In the model, Sturgeon River has a lat/lon of 50.1721, -91.5427. 

In [9]:
# selecting the value of model lat/lon from the station dataframe
latstn = station_df[station_df["STATION_NUMBER"] == stn_num]["MODEL_LATITUDE"].values[0]
lonstn = station_df[station_df["STATION_NUMBER"] == stn_num]["MODEL_LONGITUDE"].values[0]

Now, let's search for the forecast using the model lat and lon to define the subsets. Note: the next cell may take a few minutes to run because several requests are being made.

In [ ]:
# create WCS object
wcs = WebCoverageService(
    f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name_fcast}', 
    auth=Authentication(username=login['Username'], password=login['Password']),
    version='2.0.1',
    timeout=300
)

# local time
# modify as needed for the time zone of the station of interest (remember to adjust for daylight's savings). The forecast is given in UTC.
time_zone = -5

# for each forecast hour, make a WCS request
fcst_arrys = []

for i, hr in enumerate(fcasthrs_str):
    response = wcs.getCoverage(
        identifier = [layer_name_fcast], 
        format = 'image/netcdf', 
        subsettingcrs = 'EPSG:4326', 
        subsets = [('lat', latstn-0.5, latstn+0.5), ('lon', lonstn-0.5, lonstn+0.5)], 
        resolutions=[('lat', 0.008333), ('lon', 0.008333)],
        DIM_REFERENCE_TIME=newest_fcast, 
        TIME=hr 
    )

    # read into an xarray
    ds = xr.open_dataset(response.read()).load()
    
    # add the time metadata as a new dimension and coordinate
    ds = ds.expand_dims(time=[fcasthrs[i] + timedelta(hours=time_zone)])
    
    # append to list of xarrays
    fcst_arrys.append(ds)
    
fcasts = xr.concat([ds for ds in fcst_arrys], dim='time')

In [11]:
# checking to see that the time and lat/lon dimensions look sensible
fcasts.head()

<xarray.Dataset> Size: 625B
Dimensions:         (time: 5, lat: 5, lon: 5)
Coordinates:
  * time            (time) datetime64[ns] 40B 2025-08-04T20:00:00 ... 2025-08-05
  * lat             (lat) float64 40B 49.68 49.68 49.69 49.7 49.71
  * lon             (lon) float64 40B -92.04 -92.03 -92.02 -92.01 -92.01
Data variables:
    crs             (time) |S1 5B b'' b'' b'' b'' b''
    RiverDischarge  (time, lat, lon) float32 500B 0.00273 0.0 0.0 ... 0.0 0.0
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Tue Aug 05 15:31:33 2025: GDAL CreateCopy( ...

Notice that there is a data variable called 'crs'. To be able to plot 'RiverDischarge' properly, we want to remove this from our data.

In [12]:
# remove crs data variable to work with RiverDischarge properly
fcasts = fcasts.drop_vars('crs')
fcasts.head()

<xarray.Dataset> Size: 620B
Dimensions:         (time: 5, lat: 5, lon: 5)
Coordinates:
  * time            (time) datetime64[ns] 40B 2025-08-04T20:00:00 ... 2025-08-05
  * lat             (lat) float64 40B 49.68 49.68 49.69 49.7 49.71
  * lon             (lon) float64 40B -92.04 -92.03 -92.02 -92.01 -92.01
Data variables:
    RiverDischarge  (time, lat, lon) float32 500B 0.00273 0.0 0.0 ... 0.0 0.0
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Tue Aug 05 15:31:33 2025: GDAL CreateCopy( ...

Now, let's modify the forecast to select only the grid cell with the station of interest

In [13]:
# function to relate the CCMEP station location to its grid location
def find_stn_on_grid(fcasts, latstn, lonstn):
    
    # find grid lat
    closest_lat_idx = (np.abs(fcasts.lat.data - latstn)).argmin()
    lat = fcasts.lat.data[closest_lat_idx]
    
    # find grid lon
    closest_lon_idx = (np.abs(fcasts.lon.data - lonstn)).argmin()
    lon = fcasts.lon.data[closest_lon_idx]
    
    return lat, lon

In [14]:
# find the station on the grid
lat, lon = find_stn_on_grid(fcasts, latstn, lonstn)

# compare the CCMEP station location and the grid station location
ll_info = f"CCMEP lat/lon ({str(latstn)}, {str(lonstn)}) | Grid lat/lon ({str(np.round(lat,4))}, {str(np.round(lon,4))})"
ll_info

'CCMEP lat/lon (50.1721, -91.5427) | Grid lat/lon (50.1679, -91.5469)'

In [15]:
# select the grid cell of interest
station = fcasts.sel(lat=lat, lon=lon, method='nearest')

# plot the data
forecast_plot = station.hvplot(title='Sturgeon River at McDougall Mills | '+ll_info, ylabel='River Discharge (m3/s)', width=900, 
                               label='Latest Forecast', line_dash = 'dashed', color='royalblue')
forecast_plot

:Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)

## Comparing the analysis to observations

To compare DHPS data to observations, we now want to look at an analysis layer.

In [ ]:
# now we want an analysis layer to compare observations to
layer_name_ana = 'DHPS-Analysis_1km_RiverDischarge'

# first querying the WMS for time metadata
wms = WebMapService(
    f'https://geo.weather.gc.ca/geomet?&SERVICE=WMS&LAYERS={layer_name_ana}',
    version='1.3.0',
    auth=Authentication(username=login['Username'], password=login['Password']),
    timeout=300
)

In [17]:
# define time variables
first_analysis, last_analysis, timestep = wms[layer_name_ana].dimensions['time']['values'][0].split('/')

print(wms[layer_name_ana].dimensions['time']['values'][0].split('/'))

['2025-08-02T13:00:00Z', '2025-08-05T00:00:00Z', 'PT1H']


In [18]:
# convert dates to datetime objects
first = datetime.strptime(first_analysis, iso_format)
last = datetime.strptime(last_analysis, iso_format)

# remove anything that isn't a number from the datetime interval 
intvl = int(re.sub(r'\D', '', timestep))

# create a list of analysis datetimes (we will add these to the requested data)
anahrs = [first]
while first < last:
    first = first + timedelta(hours=intvl)
    anahrs.append(first)

# create a list of iso formatted forecast datetime strings (we will use these in the WCS requests)
anahrs_str = [datetime.strftime(hr, iso_format) for hr in anahrs]

This next cell may take a few minutes to run.

In [ ]:
# create WCS object
wcs = WebCoverageService(
    f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name_ana}', 
    auth=Authentication(username=login['Username'], password=login['Password']),
    version='2.0.1',
    timeout=300
)

# for each analysis hour, make a WCS request
ana_arrys = []

for i, hr in enumerate(anahrs_str):
    response = wcs.getCoverage(
        identifier = [layer_name_ana], 
        format = 'image/netcdf', 
        subsettingcrs = 'EPSG:4326', 
        subsets = [('lat', latstn-0.5, latstn+0.5), ('lon', lonstn-0.5, lonstn+0.5)], 
        resolutions=[('lat', 0.008333), ('lon', 0.008333)],
        TIME=hr 
    )

    # read into an xarray
    ds = xr.open_dataset(response.read()).load()
    
    # add the time metadata as a new dimension and coordinate
    # change to local time
    ds = ds.expand_dims(time=[anahrs[i] + timedelta(hours=time_zone)])
    
    # append to list of xarrays
    ana_arrys.append(ds)
    
analysis = xr.concat([ds for ds in ana_arrys], dim='time')

In [20]:
# remove crs data variable to work with RiverDischarge properly
analysis = analysis.drop_vars('crs')
analysis

<xarray.Dataset> Size: 3MB
Dimensions:         (time: 60, lat: 120, lon: 120)
Coordinates:
  * time            (time) datetime64[ns] 480B 2025-08-02T08:00:00 ... 2025-0...
  * lat             (lat) float64 960B 49.68 49.68 49.69 ... 50.65 50.66 50.67
  * lon             (lon) float64 960B -92.04 -92.03 -92.02 ... -91.06 -91.05
Data variables:
    RiverDischarge  (time, lat, lon) float32 3MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Tue Aug 05 15:33:18 2025: GDAL CreateCopy( ...

Select the grid cell corresponding to our station of interest.

In [21]:
# select the grid cell of interest for the analysis
station_ana = analysis.sel(lat=lat, lon=lon, method='nearest')
station_ana

<xarray.Dataset> Size: 736B
Dimensions:         (time: 60)
Coordinates:
  * time            (time) datetime64[ns] 480B 2025-08-02T08:00:00 ... 2025-0...
    lat             float64 8B 50.17
    lon             float64 8B -91.55
Data variables:
    RiverDischarge  (time) float32 240B 20.67 20.57 20.57 ... 20.13 20.24 19.61
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Tue Aug 05 15:33:18 2025: GDAL CreateCopy( ...

### Obtaining observations

Now, let's grab the observations from the station and see how they compare. We can get this information from [OGC-Features API](https://eccc-msc.github.io/open-data/msc-geomet/ogc_api_en/).

In [22]:
# define bounding box
bbox = [
    lon - 0.5,
    lat - 0.5,
    lon + 0.5,
    lat + 0.5,
]

# convert bbox to strings otherwise you will get an error when querying the station data
bbox_str = [str(x) for x in bbox]

Let's first look at information about the station.

In [23]:
# creating a connection to the OGC-Features API and making a query
oafeat = Features("https://api.weather.gc.ca/")

station_data = oafeat.collection_items(
    "hydrometric-stations", 
    bbox=bbox_str, 
    STATUS_EN="Active" #selecting only active stations
)

In [24]:
# we can view this as a table
station_table = gpd.GeoDataFrame.from_features(station_data['features'])
station_table.head(10)

,geometry,STATION_NAME,IDENTIFIER,STATION_NUMBER,PROV_TERR_STATE_LOC,STATUS_EN,STATUS_FR,CONTRIBUTOR_EN,CONTRIBUTOR_FR,VERTICAL_DATUM,REAL_TIME,RHBN,DRAINAGE_AREA_GROSS,DRAINAGE_AREA_EFFECT
0,POINT (-91.45992 49.87339),ENGLISH RIVER AT UMFREVILLE,05QA002,05QA002,ON,Active,En service,LAKE OF THE WOODS CONTROL BOARD,COMMISSION DE CONTROLE DU LAC DES BOIS,GEODETIC SURVEY OF CANADA DATUM (LOCAL 1923 ADJ.),1,0,6230.0,None
1,POINT (-91.54075 50.16728),STURGEON RIVER AT MCDOUGALL MILLS,05QA004,05QA004,ON,Active,En service,LAKE OF THE WOODS CONTROL BOARD,COMMISSION DE CONTROLE DU LAC DES BOIS,ASSUMED DATUM,1,0,4440.0,None
2,POINT (-91.91289 50.09214),PELICAN LAKE AT SIOUX LOOKOUT,05QA006,05QA006,ON,Active,En service,LAKE OF THE WOODS CONTROL BOARD,COMMISSION DE CONTROLE DU LAC DES BOIS,ASSUMED DATUM,1,0,NaN,None


Notice our bounding box also captured some surronding stations.

Now, lets search for real-time discharge data and select only for the station of interest. It is important to note that the realtime data is preliminary and has not been reviewed for quality. For more information, refer to the [Disclaimer for Hydrometric Information](https://wateroffice.ec.gc.ca/disclaimer_info_e.html).

In [25]:
# define start and end times for observations corresponding to the analysis times
start_date = anahrs_str[0]
end_date = anahrs_str[-1]

In [ ]:
# Retrieval of real-time data
hydro_data = oafeat.collection_items(
    'hydrometric-realtime',
    bbox=bbox_str,
    datetime=f"{start_date}/{end_date}",
    STATION_NUMBER = stn_num # selecting only the station of interest
)

In [29]:
# turn the data into a table
hydro_table = gpd.GeoDataFrame.from_features(hydro_data['features'])
hydro_table.head()

,geometry,IDENTIFIER,STATION_NUMBER,STATION_NAME,PROV_TERR_STATE_LOC,DATETIME,DATETIME_LST,LEVEL,DISCHARGE,LEVEL_SYMBOL_EN,LEVEL_SYMBOL_FR,DISCHARGE_SYMBOL_EN,DISCHARGE_SYMBOL_FR
0,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:00:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:00:00Z,2025-08-02T08:00:00-05:00,28.818,20.6,None,None,None,None
1,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:05:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:05:00Z,2025-08-02T08:05:00-05:00,28.818,20.6,None,None,None,None
2,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:10:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:10:00Z,2025-08-02T08:10:00-05:00,28.819,20.6,None,None,None,None
3,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:15:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:15:00Z,2025-08-02T08:15:00-05:00,28.818,20.6,None,None,None,None
4,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:20:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:20:00Z,2025-08-02T08:20:00-05:00,28.817,20.5,None,None,None,None


In [30]:
# transform the table to a dataframe for graphing
obs_df = pd.DataFrame(hydro_table)

# change the index for graphing purposes
# DATETIME_LST represents the local standard time
obs_df['DATETIME_LST'] = pd.to_datetime(obs_df['DATETIME_LST'])
obs_df = obs_df.set_index('DATETIME_LST')

obs_df.head()

,geometry,IDENTIFIER,STATION_NUMBER,STATION_NAME,PROV_TERR_STATE_LOC,DATETIME,LEVEL,DISCHARGE,LEVEL_SYMBOL_EN,LEVEL_SYMBOL_FR,DISCHARGE_SYMBOL_EN,DISCHARGE_SYMBOL_FR
DATETIME_LST,,,,,,,,,,,,
2025-08-02 08:00:00-05:00,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:00:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:00:00Z,28.818,20.6,None,None,None,None
2025-08-02 08:05:00-05:00,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:05:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:05:00Z,28.818,20.6,None,None,None,None
2025-08-02 08:10:00-05:00,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:10:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:10:00Z,28.819,20.6,None,None,None,None
2025-08-02 08:15:00-05:00,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:15:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:15:00Z,28.818,20.6,None,None,None,None
2025-08-02 08:20:00-05:00,POINT (-91.54075 50.16728),05QA004.2025-08-02T13:20:00Z,05QA004,STURGEON RIVER AT MCDOUGALL MILLS,ON,2025-08-02T13:20:00Z,28.817,20.5,None,None,None,None


Next, we want to average the observations for each hour for a better comparison to the analysis. However,  we need to add one hour to the averaged data because DHPS will label a time as 13:00:00 for data averaged from 12:00:00 to 12:59:00, but pandas would label this as 12:00:00. Thus, to match DHPS nomenclature, we must add an hour.

In [31]:
# average all data within an hour
hourly_obs = obs_df['DISCHARGE'].resample('h').mean()
# adjust time to match DHPS averaging
hourly_obs.index = hourly_obs.index + pd.Timedelta(hours=1)
# display
hourly_obs

DATETIME_LST
2025-08-02 09:00:00-05:00    20.566667
2025-08-02 10:00:00-05:00    20.400000
2025-08-02 11:00:00-05:00    20.350000
2025-08-02 12:00:00-05:00    20.275000
2025-08-02 13:00:00-05:00    20.108333
2025-08-02 14:00:00-05:00    19.941667
2025-08-02 15:00:00-05:00    19.891667
2025-08-02 16:00:00-05:00    19.966667
2025-08-02 17:00:00-05:00    19.833333
2025-08-02 18:00:00-05:00    19.808333
2025-08-02 19:00:00-05:00    19.925000
2025-08-02 20:00:00-05:00    20.125000
2025-08-02 21:00:00-05:00    20.241667
2025-08-02 22:00:00-05:00    20.250000
2025-08-02 23:00:00-05:00    20.275000
2025-08-03 00:00:00-05:00    20.250000
2025-08-03 01:00:00-05:00    20.283333
2025-08-03 02:00:00-05:00    20.300000
2025-08-03 03:00:00-05:00    20.291667
2025-08-03 04:00:00-05:00    20.233333
2025-08-03 05:00:00-05:00    20.216667
2025-08-03 06:00:00-05:00    20.241667
2025-08-03 07:00:00-05:00    20.241667
2025-08-03 08:00:00-05:00    20.283333
2025-08-03 09:00:00-05:00    20.233333
2025-08-03 1

Now graph the analysis and observations!

In [32]:
# create a plot for the analysis
analysis_plot = station_ana.hvplot(title='Sturgeon River at McDougall Mills | '+ll_info, ylabel='River Discharge (m3/s)', 
                                   width=900, label='Analysis', color='royalblue')

# create a plot for the model
obs_plot = hourly_obs.hvplot(y='DISCHARGE', width=900, color='red', label='Observations',
    kind='scatter', marker='x')

# combine the plots
obs_ana_plot = analysis_plot * obs_plot
# display the plot
obs_ana_plot

:Overlay
   .Curve.Analysis       :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)
   .Scatter.Observations :Scatter   [DATETIME_LST]   (DISCHARGE)

There can be a delay in the availability of the station observations, which is why the observations don't cover the full analysis period. 

Now, let's combine our figures of the analysis and the forecast.

In [34]:
all = obs_ana_plot * forecast_plot
all

:Overlay
   .Curve.Analysis        :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)
   .Scatter.Observations  :Scatter   [DATETIME_LST]   (DISCHARGE)
   .Curve.Latest_Forecast :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)

## Comparing previous forecasts to the analysis

It may be of interest to see how older forecasts compare to the analysis. We are able to do this by modifying the datetimes of interest based on issue_interval.

In [35]:
print("The interval between forecasts is "+issue_interval+".")

The interval between forecasts is PT12H.


We can see that the forecasts are issued every 12 hours. We can then subtract multiples of 12 hours to get previous forecasts. Here, we will get the previous two forecasts (from 12 hours and 24 hours ago).

In [36]:
# change first_datetime for older forecasts
dt = datetime.strptime(first_datetime, "%Y-%m-%dT%H:%M:%SZ")  
dt = dt.replace(tzinfo=timezone.utc)                          
# subtract 12 hours
dt_modified = dt - timedelta(hours=12)
# subtract 24 hours
dt_modified_2 = dt - timedelta(hours=24)
# change back to needed format
first_dt_old_fcast = dt_modified.strftime("%Y-%m-%dT%H:%M:%SZ")
first_dt_old_fcast_2 = dt_modified_2.strftime("%Y-%m-%dT%H:%M:%SZ")

# do also for last_datetime
dt = datetime.strptime(last_datetime, "%Y-%m-%dT%H:%M:%SZ")  
dt = dt.replace(tzinfo=timezone.utc)                          
# subtract 12 hours
dt_modified = dt - timedelta(hours=12)
# subtract 24 hours
dt_modified_2 = dt - timedelta(hours=24)
# change back to needed format
last_dt_old_fcast = dt_modified.strftime("%Y-%m-%dT%H:%M:%SZ")
last_dt_old_fcast_2 = dt_modified_2.strftime("%Y-%m-%dT%H:%M:%SZ")

In [37]:
# we also need to change newest_fcast, which was used as the reference time during the forecast query
dt = datetime.strptime(newest_fcast, "%Y-%m-%dT%H:%M:%SZ")  
dt = dt.replace(tzinfo=timezone.utc)                          
# subtract 12 hours
dt_modified = dt - timedelta(hours=12)
# subtract 24 hours
dt_modified_2 = dt - timedelta(hours=24)
# change back to needed format
ref_fcast = dt_modified.strftime("%Y-%m-%dT%H:%M:%SZ")
ref_fcast_2 = dt_modified_2.strftime("%Y-%m-%dT%H:%M:%SZ")

Use these modified time variables to query the forecast.

In [38]:
# convert dates to datetime objects
# 12 hour ago forecast
first = datetime.strptime(first_dt_old_fcast, iso_format)
last = datetime.strptime(last_dt_old_fcast, iso_format)

# 24 hour ago forecast
first_2 = datetime.strptime(first_dt_old_fcast_2, iso_format)
last_2 = datetime.strptime(last_dt_old_fcast_2, iso_format)

# remove anything that isn't a number from the datetime interval (time between forecasts)
intvl = int(re.sub(r'\D', '', datetime_interval))

# create a list of forecast datetimes (we will add these to the requested data)
fcasthrs = [first]
while first < last:
    first = first + timedelta(hours=intvl)
    fcasthrs.append(first)

fcasthrs_2 = [first_2]
while first_2 < last_2:
    first_2 = first_2 + timedelta(hours=intvl)
    fcasthrs_2.append(first_2)

# create a list of iso formatted forecast datetime strings (we will use these in the WCS requests)
fcasthrs_str = [datetime.strftime(hr, iso_format) for hr in fcasthrs]
fcasthrs_str_2 = [datetime.strftime(hr, iso_format) for hr in fcasthrs_2]

The following two cells may take a minute to run.

In [ ]:
# create WCS object
wcs = WebCoverageService(
    f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name_fcast}', 
    auth=Authentication(username=login['Username'], password=login['Password']),
    version='2.0.1',
    timeout=300
)

# doing this for the 12 hour ago forecast
# for each forecast hour, make a WCS request
fcst_arrys = []

for i, hr in enumerate(fcasthrs_str):
    response = wcs.getCoverage(
        identifier = [layer_name_fcast], 
        format = 'image/netcdf', 
        subsettingcrs = 'EPSG:4326', 
        subsets = [('lat', latstn-0.5, latstn+0.5), ('lon', lonstn-0.5, lonstn+0.5)], 
        resolutions=[('lat', 0.008333), ('lon', 0.008333)],
        DIM_REFERENCE_TIME=ref_fcast, 
        TIME=hr 
    )

    # read into an xarray
    ds = xr.open_dataset(response.read()).load()
    
    # add the time metadata as a new dimension and coordinate
    ds = ds.expand_dims(time=[fcasthrs[i] + timedelta(hours=time_zone)])
    
    # append to list of xarrays
    fcst_arrys.append(ds)
    
older_fcasts = xr.concat([ds for ds in fcst_arrys], dim='time')

In [ ]:
# doing this now for the 24 hours ago forecast
# make a WCS request for the second forecast of interest
fcst_arrys = []

for i, hr in enumerate(fcasthrs_str_2):
    response = wcs.getCoverage(
        identifier = [layer_name_fcast], 
        format = 'image/netcdf', 
        subsettingcrs = 'EPSG:4326', 
        subsets = [('lat', latstn-0.5, latstn+0.5), ('lon', lonstn-0.5, lonstn+0.5)], 
        resolutions=[('lat', 0.008333), ('lon', 0.008333)],
        DIM_REFERENCE_TIME=ref_fcast_2, # changed to oldest_fcst
        TIME=hr 
    )

    # read into an xarray
    ds = xr.open_dataset(response.read()).load()
    
    # add the time metadata as a new dimension and coordinate
    ds = ds.expand_dims(time=[fcasthrs_2[i] + timedelta(hours=time_zone)])
    
    # append to list of xarrays
    fcst_arrys.append(ds)
    
older_fcasts_2 = xr.concat([ds for ds in fcst_arrys], dim='time')

In [41]:
# remove crs data variable to work with RiverDischarge properly
older_fcasts = older_fcasts.drop_vars('crs')
older_fcasts_2 = older_fcasts_2.drop_vars('crs')

In [42]:
# select the grid cell of interest
older_fcast_station = older_fcasts.sel(lat=lat, lon=lon, method='nearest')
older_fcast_station_2 = older_fcasts_2.sel(lat=lat, lon=lon, method='nearest')

In [43]:
# plot the data
oldest_forecast_plot = older_fcast_station.hvplot(title='Sturgeon River at McDougall Mills | '+ll_info, ylabel='River Discharge (m3/s)', width=900, 
                               label='Previous Forecast (-12h)', line_dash = 'dotted', color='gray')
oldest_forecast_plot2 = older_fcast_station_2.hvplot(title='Sturgeon River at McDougall Mills | '+ll_info, ylabel='River Discharge (m3/s)', width=900, 
                               label='Previous Forecasts (-24h)', line_dash = 'dotted', color='black')
# now combine with our other plot
all*oldest_forecast_plot*oldest_forecast_plot2

:Overlay
   .Curve.Analysis                                                               :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)
   .Scatter.Observations                                                         :Scatter   [DATETIME_LST]   (DISCHARGE)
   .Curve.Latest_Forecast                                                        :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)
   .Curve.Previous_Forecast_left_parenthesis_hyphen_minus_12h_right_parenthesis  :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)
   .Curve.Previous_Forecasts_left_parenthesis_hyphen_minus_24h_right_parenthesis :Curve   [time]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)

This figure is interactive, so feel free to zoom into any times of interest for easier viewing.